# XGBoost 再学習ノートブック v4

| セル | 内容 |
|------|------|
| セル1 | Google Drive マウント + セットアップ |
| セル2 | src/ 強制アップデート（GitHub最新） |
| セル3 | 展開予測モデル再学習（19特徴量ペースモデル） |
| セル4 | speed_index キャッシュ再構築 + 学習データ生成 |
| セル5 | XGB再学習（通常モデル） + キャリブレーション |
| セル6 | 残差学習モデル（市場コピー排除実験） |
| セル7 | Temperature Scaling + モデル比較 |
| セル8 | 統合テスト（cal_prob合計・重要度確認） |
| セル9 | 再学習モデルを GitHub main にプッシュ（本番反映） |

### v3 からの変更点
- セル3追加: 展開予測モデル再学習（19特徴量ペースモデル + パーセンタイル分類）
- セル6追加: 残差学習モデル（f_popularity除外 + base_margin）
- 学習データ日付: train_end=2026-06-30, val=2026-07-01〜07-13 に更新
- pairwise モデル学習を削除（大掃除で廃止済み）
- 強制アップデートに train_pace_model.py / market_correction.py / shap_diagnosis.py 追加

> 使い方: セル1〜8を順に実行。結果に満足したらセル9で本番反映。
> 前提: Colabシークレットに GITHUB_PAT を登録し、「ノートブックからのアクセス」をON。

In [ ]:
# == セル1: セットアップ ===================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
# == セル2: src/ 強制アップデート（GitHub最新コードを取得）==================
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())
files = [
    # tools
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/build_training_data.py',
    'src/tools/train_xgb.py',
    'src/tools/calibrate_xgb.py',
    'src/tools/generate_style_advantage.py',
    'src/tools/train_pace_model.py',
    'src/tools/shap_diagnosis.py',
    # features
    'src/features/engine.py',
    'src/features/speed_index.py',
    'src/features/market_correction.py',
    # utils
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    # scraper
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    # models
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/calibration_xgb.py',
    'src/models/predict.py',
    # betting
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
    'src/betting/race_simulator.py',
    'src/betting/ev_calculator.py',
    'src/betting/rating_calibration.py',
    'src/betting/payout_estimator.py',
    'src/betting/dual_model.py',
    'src/betting/bet_optimizer.py',
    'src/betting/shadow.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
    print(f'OK {rel}')

# data/*.json も最新版に更新
for rel_data in ['data/course_profiles.json', 'data/note_schema.json']:
    dest = f'{BASE_DIR}/{rel_data}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{rel_data}?nocache={_cb}', dest)
        print(f'OK {rel_data}')
    except Exception as e:
        print(f'SKIP {rel_data}: {e}')

# モジュールキャッシュクリア
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# 主要機能の存在確認
with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_speed_fig_last', 'f_popularity', 'f_style_course_fit',
            'predict_race_pace', '_build_pace_features_for_inference',
            '_JOCKEY_PACE_STATS', '_XGB_RESIDUAL']:
    _ok = _kw in _eng_src
    print(f'  {"OK" if _ok else "NG"} engine.py: {_kw}')

with open(f'{BASE_DIR}/src/tools/train_pace_model.py') as _f:
    _pm_src = _f.read()
for _kw in ['_build_pace_percentiles', '_PACE_PERCENTILE_CACHE', '_dist_zone']:
    _ok = _kw in _pm_src
    print(f'  {"OK" if _ok else "NG"} train_pace_model.py: {_kw}')

print('\ndone')

In [ ]:
# == セル3: 展開予測モデル再学習（19特徴量ペースモデル）====================
# ペース分類をパーセンタイルベースに改善 + 19特徴量で学習
# -> data/pace_model.pkl + data/jockey_pace_stats.json が生成される
# -> 次のセルでbuild_training_dataが参照し、XGBの特徴量にも反映
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_pace_model import train_pace_model

pace_result = train_pace_model(BASE_DIR)
print(f'\n== ペースモデル結果 ==')
print(f'精度: {pace_result.get("accuracy", "N/A")}')
print(f'分布: {pace_result.get("distribution", "N/A")}')
print(f'特徴量数: {pace_result.get("n_features", "N/A")}')

# 検証: 分布が偏っていないか
dist = pace_result.get('distribution', {})
if dist:
    vals = list(dist.values())
    max_ratio = max(vals) / max(min(vals), 0.01)
    if max_ratio > 5:
        print(f'\n!! 分布が偏っています (max/min={max_ratio:.1f})。閾値を確認してください')
    else:
        print(f'\nOK 分布バランス良好 (max/min={max_ratio:.1f})')

In [ ]:
# == セル4: speed_index キャッシュ再構築 + 学習データ生成 ===================
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.features.speed_index import rebuild_speed_index_cache
rebuild_speed_index_cache(BASE_DIR)

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# コース×脚質の style_advantage を history.db 実データで更新
from src.tools.generate_style_advantage import generate_style_advantage
generate_style_advantage(BASE_DIR)

# 学習データ生成（全特徴量入りCSV）
from src.tools.build_training_data import build_training_data
df_all = build_training_data(BASE_DIR)

# 特徴量充足率確認
print('\n== 特徴量充足率 ==')
CHECK_COLS = [
    'f_popularity', 'f_pop_last', 'f_pop_avg', 'f_beat_market_rate',
    'f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max',
    'f_style_course_fit', 'f_pace_fit', 'f_style_total_fit',
    'f_finish_time_avg', 'f_time_diff_avg',
    'f_member_level_avg',
]
for c in CHECK_COLS:
    if c in df_all.columns:
        pct = 100 * df_all[c].notna().sum() / len(df_all)
        mark = 'OK' if pct > 50 else 'LOW'
        print(f'  {mark} {c}: {pct:.1f}%')
    else:
        print(f'  NG {c}: 列なし')
print(f'\n総列数: {len(df_all.columns)}  総行数: {len(df_all)}')

In [ ]:
# == セル5: XGB再学習（通常モデル） + キャリブレーション ====================
# 日付は history.db のデータ範囲に合わせて調整してください
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_xgb import train_xgb
from src.tools.calibrate_xgb import run_xgb_calibration

# 学習: 〜6月末, 検証: 7月
metrics = train_xgb(BASE_DIR,
    train_end='2026-06-30',
    val_start='2026-07-01',
    val_end='2026-07-13')

print(f'\n== 再学習結果 ==')
print(f'AUC: {metrics.get("auc", "N/A")}')
print(f'旧AUC: {metrics.get("old_auc", "N/A")}')
print(f'特徴量数: {metrics.get("n_features", "N/A")}')

auc = metrics.get('auc', 0)
if auc >= 0.75:
    run_xgb_calibration(BASE_DIR)
    print('OK キャリブレーション完了')
else:
    print(f'!! AUC={auc:.4f} < 0.75 -> キャリブレーションスキップ')

# 判定
if auc >= 0.82:
    print(f'\n>>> AUC {auc:.4f} >= 0.82: 現行水準維持。セル9でpush可能')
elif auc >= 0.80:
    print(f'\n>>> AUC {auc:.4f}: やや低下。feature_importanceを確認してからpush判断')
else:
    print(f'\n>>> AUC {auc:.4f} < 0.80: 問題あり。pushしないこと')

In [ ]:
# == セル6: 残差学習モデル（市場コピー排除実験）============================
# f_popularity を特徴量から除外し、logit(p_market) を base_margin として渡す。
# モデルは「市場からのズレ」だけを学習 -> AI独自の予測力を測定できる。
# 結果は別ファイルに保存 -> 現行モデルには影響しない。
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_xgb import train_xgb

res_metrics = train_xgb(BASE_DIR,
    train_end='2026-06-30',
    val_start='2026-07-01',
    val_end='2026-07-13',
    residual=True)

print(f'\n== 残差学習結果 ==')
print(f'残差AUC: {res_metrics.get("auc", "N/A")}')
print(f'通常AUC: {metrics.get("auc", "N/A")}  (セル5の結果)')

res_auc = res_metrics.get('auc', 0)
normal_auc = metrics.get('auc', 0)
if res_auc >= normal_auc * 0.95:
    print(f'\n>>> 残差モデルが通常の95%以上: AI独自の予測力あり')
    print('    本番切替する場合はセル9のFILESに_residualファイルを追加')
else:
    gap = (1 - res_auc / max(normal_auc, 0.01)) * 100
    print(f'\n>>> 残差モデルが通常より{gap:.1f}%低い: 予測力の大半は市場コピー')
    print('    現行モデルを維持')

In [ ]:
# == セル7: Temperature Scaling + ランキングモデル(ndcg) ====================
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# B2 ランキングモデル学習（rank:ndcg）
from src.tools.train_ranking_model import train_ranking_model

print('== rank:ndcg 学習 ==')
train_ranking_model(BASE_DIR, objective='rank:ndcg', model_suffix='ndcg',
                    train_end='2026-06-30',
                    val_start='2026-07-01', val_end='2026-07-13')

# Temperature Scaling
print('\n== Temperature Scaling ==')
from src.betting.rating_calibration import calibrate_all_models
calibrate_all_models(BASE_DIR,
                     val_start='2026-05-01',
                     val_end='2026-06-30',
                     n_sims=5000)

# モデル比較
print('\n== モデル比較 ==')
from src.tools.compare_models import compare_all_models, print_comparison
comp = compare_all_models(BASE_DIR, [
    ('2026-07-01', '2026-07-13'),
    ('2026-06-01', '2026-06-30'),
], n_sims=5000)
print_comparison(comp)

In [ ]:
# == セル8: 統合テスト =====================================================
import pickle, json, numpy as np, pandas as pd

with open(f'{BASE_DIR}/data/xgb_fukusho_model.pkl', 'rb') as f: model = pickle.load(f)
with open(f'{BASE_DIR}/data/xgb_feature_cols.json')        as f: meta  = json.load(f)
cols = meta['feature_cols']
with open(f'{BASE_DIR}/data/xgb_calibrator.pkl', 'rb')     as f: calib = pickle.load(f)

print(f'モデル特徴量数: {len(cols)}')
print(f'学習日時: {meta.get("trained_at", "不明")}')
print(f'Val AUC: {meta.get("val_auc", "不明")}')
print(f'キャリブレーター: {type(calib).__name__}')

# ダミー16頭で cal_prob 合計確認
dummy = pd.DataFrame([{c: np.random.normal(5.0, 1.5) for c in cols} for _ in range(16)])
raw_probs = model.predict_proba(dummy)[:, 1]
cal_probs = np.array(calib.transform(raw_probs))

print(f'\n== ダミー16頭テスト ==')
print(f'  raw_prob range: {raw_probs.min():.3f} ~ {raw_probs.max():.3f}')
print(f'  cal_prob 合計:  {cal_probs.sum():.3f}  (2.8~3.2 が正常)')
cal_ok = 2.5 <= cal_probs.sum() <= 3.5
var_ok = raw_probs.max() - raw_probs.min() > 0.2
print(f'  合計チェック: {"OK" if cal_ok else "NG"}')
print(f'  分散チェック: {"OK" if var_ok else "NG"}')

# 特徴量重要度トップ20
HIGHLIGHT = {
    'f_popularity': 'MARKET', 'f_pop_last': 'MARKET', 'f_pop_avg': 'MARKET',
    'f_beat_market_rate': 'MARKET',
    'f_speed_fig_last': 'SPEED', 'f_speed_fig_avg': 'SPEED', 'f_speed_fig_max': 'SPEED',
    'f_pace': 'PACE',
}
imps = sorted(zip(cols, model.feature_importances_), key=lambda x: x[1], reverse=True)[:20]
print('\n== 特徴量重要度 Top 20 ==')
for name, imp in imps:
    tag = HIGHLIGHT.get(name, '')
    print(f'  {name:<42} {imp*100:5.2f}%  {tag}')

# 市場特徴量の合計重要度
mkt_imp = sum(imp for n, imp in zip(cols, model.feature_importances_)
              if n in ('f_popularity', 'f_pop_last', 'f_pop_avg', 'f_beat_market_rate'))
print(f'\n市場特徴量の合計重要度: {mkt_imp*100:.1f}%')
if mkt_imp > 0.30:
    print('  -> 30%超: 市場依存が強い。残差学習(セル6)の結果も参考に')
elif mkt_imp > 0.15:
    print('  -> 15-30%: 市場情報を適度に活用')
else:
    print('  -> 15%未満: 市場情報の取り込みが弱い可能性')

# 残差モデルがあれば比較表示
import os
res_path = f'{BASE_DIR}/data/xgb_feature_cols_residual.json'
if os.path.exists(res_path):
    with open(res_path) as f:
        res_meta = json.load(f)
    print(f'\n== 残差モデル情報 ==')
    print(f'  残差AUC: {res_meta.get("val_auc", "不明")}')
    print(f'  特徴量数: {len(res_meta.get("feature_cols", []))}')
    print(f'  residualフラグ: {res_meta.get("residual", False)}')

In [ ]:
# == セル9: 再学習モデルを GitHub main にプッシュ ===========================
# これを実行しないと、Driveで再学習したモデルが本番(GitHub Actions予想)に反映されない。
# 前提: Colabシークレットに GITHUB_PAT を登録し、ノートの「ノートブックからの
#       アクセス」をONにしておくこと。
import requests, base64, json as _json, os as _os
from google.colab import userdata

GITHUB_PAT = userdata.get('GITHUB_PAT')
REPO  = 'hanagenuku/keiba_ai'

# pushするファイル一覧（存在するもののみ）
FILES = [
    'data/xgb_fukusho_model.pkl',
    'data/xgb_feature_cols.json',
    'data/xgb_calibrator.pkl',
    'data/xgb_ranking_ndcg.pkl',
    'data/xgb_ranking_feature_cols.json',
    'data/rating_temperature.json',
    'data/pace_model.pkl',
    'data/jockey_pace_stats.json',
    'data/speed_index_cache.pkl',
]

def push_file(relpath, pat, message):
    url = f'https://api.github.com/repos/{REPO}/contents/{relpath}'
    h = {'Authorization': f'token {pat}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=h, params={'ref': 'main'})
    sha = r.json().get('sha') if r.status_code == 200 else None
    with open(f'{BASE_DIR}/{relpath}', 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    payload = {'message': message, 'content': content, 'branch': 'main'}
    if sha:
        payload['sha'] = sha
    r = requests.put(url, headers=h, json=payload)
    ok = r.status_code in (200, 201)
    status = 'OK' if ok else f'NG({r.status_code})'
    print(f'{status} {relpath}')
    if not ok:
        print(f'  -> {r.json().get("message", "")}')
    return ok

# メタ情報
with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    _meta = _json.load(f)
n = len(_meta['feature_cols'])
auc_str = f'{_meta.get("val_auc", 0):.4f}'
msg = f'model: retrain v4 {n}feat AUC={auc_str}'
print(f'コミットメッセージ: {msg}')
print(f'特徴量数: {n}')
print()

pushed = 0
for rel in FILES:
    if not _os.path.exists(f'{BASE_DIR}/{rel}'):
        print(f'SKIP（未生成）: {rel}')
        continue
    if push_file(rel, GITHUB_PAT, msg):
        pushed += 1

print(f'\n完了: {pushed}/{len(FILES)} ファイルをpush')